# Bonus: CNN zur Erkennung von Personen und Autos

Dieses Notebook implementiert ein CNN zur Erkennung von Personen und Autos und wendet es auf Bilder an, die sowohl Personen als auch Autos enthalten.

## Einführung und theoretischer Hintergrund

Die Erkennung mehrerer Objektklassen in einem Bild ist ein wichtiger Anwendungsfall im Bereich des maschinellen Sehens und der künstlichen Intelligenz. In realen Szenarien enthalten Bilder oft verschiedene Arten von Objekten, die gleichzeitig erkannt werden müssen, wie z.B. Personen und Fahrzeuge im Straßenverkehr.

In diesem Bonus-Notebook erweitern wir unsere bisherige Arbeit zur Automerkennung um die Fähigkeit, auch Personen zu erkennen. Dies demonstriert die Flexibilität und Erweiterbarkeit von CNN-basierten Objekterkennungssystemen und zeigt, wie man mehrere spezialisierte Modelle kombinieren kann, um komplexere Aufgaben zu lösen.

Für die Personenerkennung trainieren wir ein separates CNN-Modell auf dem CIFAR-10-Datensatz, ähnlich wie wir es für die Automerkennung getan haben. Der CIFAR-10-Datensatz enthält eine Kategorie "Person", die wir für das Training verwenden können. Alternativ könnten wir auch ein Multi-Klassen-Modell trainieren, das sowohl Autos als auch Personen erkennen kann, aber der Ansatz mit separaten Modellen bietet mehrere Vorteile:

1. **Modularität**: Jedes Modell kann unabhängig trainiert, optimiert und aktualisiert werden
2. **Spezialisierung**: Jedes Modell kann sich auf die spezifischen Merkmale einer Objektklasse konzentrieren
3. **Flexibilität**: Neue Objektklassen können hinzugefügt werden, ohne bestehende Modelle neu trainieren zu müssen
4. **Parallelisierung**: Die Erkennung verschiedener Objektklassen kann parallel durchgeführt werden

Nach dem Training der Modelle implementieren wir einen verbesserten Erkennungsalgorithmus, der beide Modelle verwendet, um Personen und Autos in Bildern zu erkennen. Wir verwenden dabei ähnliche Techniken wie im vorherigen Notebook, einschließlich Region Proposals, Multi-Scale-Erkennung und Non-Maximum Suppression, passen sie jedoch an, um mit mehreren Objektklassen umzugehen.

Die Ergebnisse werden visualisiert, wobei verschiedene Farben für verschiedene Objektklassen verwendet werden, um die Erkennungsergebnisse klar darzustellen.

## Überblick über die Schritte
- Laden und Vorbereiten des Datensatzes für Personenerkennung
- Training eines CNN-Modells für Personenerkennung
- Laden des vortrainierten Autoerkennungsmodells
- Implementierung einer verbesserten Erkennungsmethode basierend auf dem 05-Notebook
- Anwendung auf Testbilder mit Personen und Autos
- Visualisierung der Ergebnisse

## Importieren der benötigten Bibliotheken

Für die Implementierung der Personen- und Automerkennung benötigen wir verschiedene Python-Bibliotheken. Die meisten dieser Bibliotheken haben wir bereits in den vorherigen Notebooks verwendet, aber hier fügen wir einige zusätzliche Funktionalitäten hinzu, um mit mehreren Objektklassen umzugehen.

Die wichtigsten Bibliotheken sind:
- **tensorflow** und **keras**: Für das Training und die Verwendung der CNN-Modelle
- **numpy**: Für effiziente numerische Operationen
- **matplotlib**: Für die Visualisierung der Daten und Ergebnisse
- **PIL**: Für die Bildverarbeitung
- **skimage**: Für die Extraktion von HOG-Features und andere Bildverarbeitungsfunktionen
- **requests**: Für das Herunterladen von Testbildern aus dem Internet

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import os
import requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
import time
from skimage.feature import hog
from skimage import exposure
import random

2025-04-05 20:54:01.329799: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-05 20:54:01.330517: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-05 20:54:01.332918: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-05 20:54:01.338870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743879241.350563 1589285 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743879241.35

## Vorbereitung der Verzeichnisse

Bevor wir mit der Implementierung beginnen, erstellen wir Verzeichnisse für die Speicherung der Modelle, Testbilder und Ergebnisse. Eine gute Organisation der Projektstruktur ist wichtig für die Nachvollziehbarkeit und Wiederverwendbarkeit des Codes.

Wir erstellen spezifische Verzeichnisse für die Personenerkennung, zusätzlich zu den bereits vorhandenen Verzeichnissen für die Automerkennung. Dies ermöglicht eine klare Trennung der Modelle und Ergebnisse für die verschiedenen Objektklassen.

In [ ]:
# Vorbereitung der Verzeichnisse
data_dir = '../data'
models_dir = '../models'
keras_models_dir = os.path.join(models_dir, 'keras')
human_models_dir = os.path.join(models_dir, 'human')
test_images_dir = '../test_images'
results_dir = '../results'
human_results_dir = os.path.join(results_dir, 'human')

os.makedirs(data_dir, exist_ok=True)
os.makedirs(human_models_dir, exist_ok=True)
os.makedirs(test_images_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
os.makedirs(human_results_dir, exist_ok=True)

## Laden und Vorbereiten des CIFAR-10-Datensatzes

Der CIFAR-10-Datensatz enthält 60.000 Farbbilder in 10 Klassen, darunter die Klasse "Person" (Index 2). Wir laden den Datensatz und bereiten ihn für das Training des Personenerkennungsmodells vor.

Ähnlich wie bei der Automerkennung erstellen wir einen binären Klassifikationsdatensatz, bei dem die Klasse "Person" als positive Klasse (1) und alle anderen Klassen als negative Klasse (0) betrachtet werden. Dies ermöglicht es uns, ein spezialisiertes Modell für die Personenerkennung zu trainieren.

Die Bilder werden normalisiert, indem die Pixelwerte auf den Bereich [0, 1] skaliert werden, was für das Training von neuronalen Netzwerken üblich ist.

In [ ]:
# Laden des CIFAR-10-Datensatzes
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalisieren der Bilder
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Erstellen binärer Labels für Personenerkennung (Person = 1, Nicht-Person = 0)
# In CIFAR-10 hat die Klasse "Person" den Index 2
y_train_binary = (y_train == 2).astype(int)
y_test_binary = (y_test == 2).astype(int)

print(f"Trainingsbilder: {x_train.shape}")
print(f"Testbilder: {x_test.shape}")
print(f"Anzahl der Personenbilder im Trainingsdatensatz: {np.sum(y_train_binary == 1)}")
print(f"Anzahl der Personenbilder im Testdatensatz: {np.sum(y_test_binary == 1)}")

# Speichern der vorbereiteten Daten
np.save(os.path.join(data_dir, 'x_train_normalized_human.npy'), x_train)
np.save(os.path.join(data_dir, 'x_test_normalized_human.npy'), x_test)
np.save(os.path.join(data_dir, 'y_train_binary_human.npy'), y_train_binary)
np.save(os.path.join(data_dir, 'y_test_binary_human.npy'), y_test_binary)

## Visualisierung einiger Beispielbilder

Bevor wir mit dem Training des Modells beginnen, visualisieren wir einige Beispielbilder aus dem Datensatz, um ein besseres Verständnis für die Daten zu bekommen. Wir zeigen sowohl Bilder von Personen als auch von Nicht-Personen, um die Vielfalt der Daten zu verdeutlichen.

Diese Visualisierung hilft uns, die Herausforderungen bei der Personenerkennung besser zu verstehen, wie z.B. die Variabilität in Pose, Kleidung, Beleuchtung und Hintergrund. Sie gibt uns auch einen Eindruck von der Qualität und Auflösung der Bilder, was wichtig ist, um realistische Erwartungen an die Leistung des Modells zu haben.

In [ ]:
# Visualisierung einiger Beispielbilder
def visualize_examples(x_data, y_data, class_name, num_examples=5):
    # Indizes für positive und negative Beispiele finden
    positive_indices = np.where(y_data == 1)[0]
    negative_indices = np.where(y_data == 0)[0]
    
    # Zufällige Auswahl von Beispielen
    np.random.seed(42)  # Für Reproduzierbarkeit
    positive_samples = np.random.choice(positive_indices, size=num_examples, replace=False)
    negative_samples = np.random.choice(negative_indices, size=num_examples, replace=False)
    
    # Visualisierung
    plt.figure(figsize=(10, 4))
    
    # Positive Beispiele
    for i, idx in enumerate(positive_samples):
        plt.subplot(2, num_examples, i+1)
        plt.imshow(x_data[idx])
        plt.title(f"{class_name}")
        plt.axis('off')
    
    # Negative Beispiele
    for i, idx in enumerate(negative_samples):
        plt.subplot(2, num_examples, num_examples+i+1)
        plt.imshow(x_data[idx])
        plt.title(f"Nicht-{class_name}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualisierung von Beispielen für Personenerkennung
visualize_examples(x_train, y_train_binary, "Person")

## Definition des CNN-Modells für Personenerkennung

Jetzt definieren wir ein CNN-Modell für die Personenerkennung. Die Architektur ist ähnlich zu der, die wir für die Automerkennung verwendet haben, mit einigen Anpassungen, um die spezifischen Merkmale von Personen besser zu erfassen.

Die Architektur besteht aus:
1. **Faltungsschichten (Convolutional Layers)**: Extrahieren räumliche Merkmale aus den Bildern
2. **Pooling-Schichten (Pooling Layers)**: Reduzieren die räumliche Dimension und machen das Modell robuster gegenüber kleinen Transformationen
3. **Dropout-Schichten (Dropout Layers)**: Verhindern Overfitting durch zufälliges Deaktivieren von Neuronen während des Trainings
4. **Vollständig verbundene Schichten (Fully Connected Layers)**: Kombinieren die extrahierten Merkmale für die endgültige Klassifikation

Das Modell verwendet die binäre Kreuzentropie als Verlustfunktion und den Adam-Optimierer, der für seine gute Leistung bei einer Vielzahl von Problemen bekannt ist.

In [ ]:
# Definition des CNN-Modells für Personenerkennung
def create_person_detection_model(input_shape=(32, 32, 3)):
    model = Sequential([
        # Erste Faltungsschicht
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Zweite Faltungsschicht
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Dritte Faltungsschicht
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Flatten und vollständig verbundene Schichten
        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')  # Binäre Klassifikation (Person vs. Nicht-Person)
    ])
    
    # Kompilieren des Modells
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Erstellen des Modells
person_model = create_person_detection_model()
person_model.summary()

## Training des Personenerkennungsmodells

Jetzt trainieren wir das CNN-Modell für die Personenerkennung auf dem vorbereiteten CIFAR-10-Datensatz. Wir verwenden Callbacks für Early Stopping und Model Checkpointing, um das Training zu optimieren und das beste Modell zu speichern.

Early Stopping beendet das Training, wenn sich die Validierungsgenauigkeit über mehrere Epochen nicht verbessert, was Overfitting verhindert und Rechenzeit spart. Model Checkpointing speichert das Modell mit der besten Validierungsgenauigkeit während des Trainings.

Wir verwenden einen Teil des Trainingsdatensatzes als Validierungsdaten, um die Generalisierungsfähigkeit des Modells während des Trainings zu überwachen. Dies hilft uns, Overfitting zu erkennen und zu vermeiden.

In [ ]:
# Callbacks für das Training
checkpoint_path = os.path.join(human_models_dir, 'person_classifier_model.h5')
checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Training des Modells
batch_size = 64
epochs = 30
validation_split = 0.2

history = person_model.fit(
    x_train, y_train_binary,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=validation_split,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)

## Evaluierung des Personenerkennungsmodells

Nach dem Training evaluieren wir das Modell auf den Testdaten, um seine Generalisierungsfähigkeit zu bewerten. Wir berechnen die Genauigkeit, den Verlust und andere Metriken, um die Leistung des Modells zu quantifizieren.

Wir visualisieren auch den Trainingsverlauf, um zu sehen, wie sich die Genauigkeit und der Verlust während des Trainings entwickelt haben. Dies gibt uns Einblicke in den Lernprozess des Modells und hilft uns, mögliche Probleme wie Overfitting oder Unterfitting zu identifizieren.

In [ ]:
# Evaluierung des Modells auf den Testdaten
test_loss, test_accuracy = person_model.evaluate(x_test, y_test_binary)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Visualisierung des Trainingsverlaufs
plt.figure(figsize=(12, 5))

# Plot für die Genauigkeit
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Genauigkeit während des Trainings')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot für den Verlust
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Verlust während des Trainings')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(human_results_dir, 'person_model_training_history.png'))
plt.show()

## Laden des Autoerkennungsmodells

Nachdem wir das Personenerkennungsmodell trainiert haben, laden wir das vortrainierte Autoerkennungsmodell aus dem zweiten Notebook. Dieses Modell wurde bereits auf dem CIFAR-10-Datensatz trainiert, um Autos zu erkennen.

Durch die Kombination beider Modelle können wir sowohl Personen als auch Autos in Bildern erkennen. Dies demonstriert, wie spezialisierte Modelle für verschiedene Objektklassen kombiniert werden können, um komplexere Aufgaben zu lösen.

In [ ]:
# Laden des vortrainierten Autoerkennungsmodells
car_model_path = os.path.join(keras_models_dir, 'car_classifier_model.h5')

try:
    car_model = load_model(car_model_path)
    print(f"Autoerkennungsmodell erfolgreich geladen von: {car_model_path}")
    car_model.summary()
except Exception as e:
    print(f"Fehler beim Laden des Autoerkennungsmodells: {e}")
    print("Versuche, das Modell aus einem anderen Verzeichnis zu laden...")
    
    # Alternative Pfade probieren
    alternative_paths = [
        os.path.join(models_dir, 'car_classifier_model.h5'),
        os.path.join(models_dir, 'pretrained', 'mobilenet_final_model.h5')
    ]
    
    for alt_path in alternative_paths:
        try:
            car_model = load_model(alt_path)
            print(f"Autoerkennungsmodell erfolgreich geladen von: {alt_path}")
            car_model.summary()
            break
        except:
            continue
    else:
        print("Konnte kein Autoerkennungsmodell laden. Bitte stellen Sie sicher, dass ein trainiertes Modell verfügbar ist.")

## Hilfsfunktionen für die Bildverarbeitung

Bevor wir mit der Implementierung des Objekterkennungsalgorithmus beginnen, definieren wir einige Hilfsfunktionen für die Bildverarbeitung. Diese Funktionen sind ähnlich zu denen, die wir im vorherigen Notebook verwendet haben, wurden jedoch angepasst, um mit mehreren Objektklassen umzugehen.

Die wichtigsten Funktionen sind:
1. `load_image_from_url`: Lädt ein Bild von einer URL
2. `load_image_from_file`: Lädt ein Bild aus einer Datei
3. `preprocess_image`: Bereitet ein Bild für die Klassifikation vor

Diese Funktionen verwenden PIL für die Bildverarbeitung, was eine bessere Kompatibilität mit verschiedenen Python-Versionen bietet als OpenCV.

In [ ]:
def load_image_from_url(url):
    """
    Lädt ein Bild von einer URL.
    
    Parameter:
    - url: URL des Bildes
    
    Rückgabe:
    - image: PIL Image-Objekt
    """
    try:
        response = requests.get(url)
        image = Image.open(BytesIO(response.content))
        return image
    except Exception as e:
        print(f"Fehler beim Laden des Bildes von URL: {e}")
        return None

def load_image_from_file(file_path):
    """
    Lädt ein Bild aus einer Datei.
    
    Parameter:
    - file_path: Pfad zur Bilddatei
    
    Rückgabe:
    - image: PIL Image-Objekt
    """
    try:
        image = Image.open(file_path)
        return image
    except Exception as e:
        print(f"Fehler beim Laden des Bildes aus Datei: {e}")
        return None

def preprocess_image(image, target_size=(32, 32)):
    """
    Bereitet ein Bild für die Klassifikation vor.
    
    Parameter:
    - image: PIL Image-Objekt
    - target_size: Zielgröße für das Bild (Höhe, Breite)
    
    Rückgabe:
    - processed_image: Vorverarbeitetes Bild als NumPy-Array mit Form (1, Höhe, Breite, Kanäle)
    """
    # Konvertieren zu RGB, falls notwendig
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Skalieren auf die Zielgröße
    image = image.resize(target_size, Image.LANCZOS)
    
    # Konvertieren zu NumPy-Array und normalisieren
    img_array = np.array(image).astype('float32') / 255.0
    
    # Erweitern der Dimensionen für das Batch
    processed_image = np.expand_dims(img_array, axis=0)
    
    return processed_image

## Implementierung des Region Proposal Algorithmus

Der Region Proposal Algorithmus ist ein wichtiger Bestandteil unseres Objekterkennungssystems. Er identifiziert potenzielle Bereiche im Bild, die Objekte enthalten könnten, und reduziert so den Suchraum für den Klassifikator.

Wir verwenden einen Sliding-Window-Ansatz mit verschiedenen Fenstergrößen, um Regionen zu generieren, die potenziell Personen oder Autos enthalten könnten. Dieser Ansatz ist flexibel und kann an verschiedene Objektgrößen und -formen angepasst werden.

Zusätzlich verwenden wir HOG-Features (Histogram of Oriented Gradients), um die Anzahl der Region Proposals zu reduzieren. HOG-Features sind besonders gut geeignet, um die Form und Struktur von Objekten zu erfassen, was sie nützlich für die Erkennung von Personen und Autos macht.

In [ ]:
def generate_region_proposals(image, window_sizes, step_size):
    """
    Generiert Region Proposals mit einem Sliding-Window-Ansatz.
    
    Parameter:
    - image: PIL Image-Objekt
    - window_sizes: Liste von Fenstergrößen (Höhe, Breite)
    - step_size: Schrittweite für das Sliding Window
    
    Rückgabe:
    - regions: Liste von Regionen als (x, y, w, h) Tupel
    """
    regions = []
    width, height = image.size
    
    for window_size in window_sizes:
        for y in range(0, height - window_size + 1, step_size):
            for x in range(0, width - window_size + 1, step_size):
                regions.append((x, y, window_size, window_size))
    
    return regions

def extract_hog_features(image_region):
    """
    Extrahiert HOG-Features aus einer Bildregion.
    
    Parameter:
    - image_region: Bildregion als NumPy-Array
    
    Rückgabe:
    - features: HOG-Features als NumPy-Array
    - hog_image: Visualisierung der HOG-Features
    """
    # Konvertieren zu Graustufen
    if len(image_region.shape) == 3 and image_region.shape[2] == 3:
        gray = np.mean(image_region, axis=2)
    else:
        gray = image_region
    
    # HOG-Features extrahieren
    features, hog_image = hog(
        gray, 
        orientations=9, 
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2), 
        visualize=True, 
        feature_vector=True
    )
    
    return features, hog_image

def filter_regions_by_hog(image, regions, threshold=0.1):
    """
    Filtert Regionen basierend auf HOG-Features.
    
    Parameter:
    - image: PIL Image-Objekt
    - regions: Liste von Regionen als (x, y, w, h) Tupel
    - threshold: Schwellenwert für die HOG-Feature-Magnitude
    
    Rückgabe:
    - filtered_regions: Gefilterte Liste von Regionen
    """
    filtered_regions = []
    image_array = np.array(image)
    
    for x, y, w, h in regions:
        # Region extrahieren
        region = image_array[y:y+h, x:x+w]
        
        # HOG-Features extrahieren
        try:
            features, _ = extract_hog_features(region)
            
            # Filtern basierend auf der Feature-Magnitude
            if np.mean(features) > threshold:
                filtered_regions.append((x, y, w, h))
        except Exception as e:
            # Ignorieren von Regionen, die Probleme verursachen
            continue
    
    return filtered_regions

## Implementierung der Multi-Klassen-Objekterkennung

Jetzt implementieren wir den Algorithmus für die Multi-Klassen-Objekterkennung, der sowohl Personen als auch Autos in Bildern erkennen kann. Der Algorithmus umfasst folgende Schritte:

1. **Region Proposals generieren**: Identifizierung potenzieller Bereiche im Bild
2. **Regionen klassifizieren**: Anwendung beider CNN-Modelle auf jede Region
3. **Filterung der Ergebnisse**: Entfernung von Regionen mit niedriger Konfidenz
4. **Non-Maximum Suppression**: Entfernung überlappender Bounding Boxes für jede Klasse

Die Non-Maximum Suppression wird für jede Objektklasse separat durchgeführt, um sicherzustellen, dass wir die besten Detektionen für jede Klasse behalten. Dies ist wichtig, da verschiedene Objektklassen unterschiedliche Charakteristiken haben und unterschiedliche Schwellenwerte erfordern können.

In [ ]:
def calculate_iou(box1, box2):
    """
    Berechnet die Intersection over Union (IoU) zwischen zwei Bounding Boxes.
    
    Parameter:
    - box1: Erste Box als (x, y, w, h) Tupel
    - box2: Zweite Box als (x, y, w, h) Tupel
    
    Rückgabe:
    - iou: Intersection over Union Wert
    """
    # Konvertieren zu (x1, y1, x2, y2) Format
    x1_1, y1_1, w1, h1 = box1
    x2_1, y2_1 = x1_1 + w1, y1_1 + h1
    
    x1_2, y1_2, w2, h2 = box2
    x2_2, y2_2 = x1_2 + w2, y1_2 + h2
    
    # Berechnen der Koordinaten des Schnittbereichs
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    # Berechnen der Fläche des Schnittbereichs
    if x2_i <= x1_i or y2_i <= y1_i:
        return 0.0  # Keine Überlappung
    
    intersection_area = (x2_i - x1_i) * (y2_i - y1_i)
    
    # Berechnen der Flächen der beiden Boxen
    box1_area = w1 * h1
    box2_area = w2 * h2
    
    # Berechnen der Union-Fläche
    union_area = box1_area + box2_area - intersection_area
    
    # Berechnen der IoU
    iou = intersection_area / union_area
    
    return iou

def non_max_suppression(boxes, scores, iou_threshold=0.5):
    """
    Führt Non-Maximum Suppression auf Bounding Boxes durch.
    
    Parameter:
    - boxes: Liste von Bounding Boxes als (x, y, w, h) Tupel
    - scores: Liste von Konfidenzwerten für jede Box
    - iou_threshold: Schwellenwert für die IoU
    
    Rückgabe:
    - selected_boxes: Liste von ausgewählten Bounding Boxes
    - selected_scores: Liste von Konfidenzwerten für die ausgewählten Boxen
    """
    # Sortieren der Boxen nach Konfidenz (absteigend)
    indices = np.argsort(scores)[::-1]
    boxes = [boxes[i] for i in indices]
    scores = [scores[i] for i in indices]
    
    selected_boxes = []
    selected_scores = []
    
    while len(boxes) > 0:
        # Die Box mit der höchsten Konfidenz auswählen
        current_box = boxes[0]
        current_score = scores[0]
        
        selected_boxes.append(current_box)
        selected_scores.append(current_score)
        
        # Entfernen der ausgewählten Box
        boxes.pop(0)
        scores.pop(0)
        
        # Filtern der verbleibenden Boxen
        i = 0
        while i < len(boxes):
            iou = calculate_iou(current_box, boxes[i])
            if iou > iou_threshold:
                # Entfernen von Boxen mit hoher Überlappung
                boxes.pop(i)
                scores.pop(i)
            else:
                i += 1
    
    return selected_boxes, selected_scores

def detect_objects(image, car_model, person_model, confidence_threshold=0.7, iou_threshold=0.5):
    """
    Erkennt Personen und Autos in einem Bild mit den trainierten CNN-Modellen.
    
    Parameter:
    - image: PIL Image-Objekt
    - car_model: Trainiertes CNN-Modell für Automerkennung
    - person_model: Trainiertes CNN-Modell für Personenerkennung
    - confidence_threshold: Schwellenwert für die Konfidenz
    - iou_threshold: Schwellenwert für die IoU bei Non-Maximum Suppression
    
    Rückgabe:
    - car_detections: Liste von Autodetektionen als (x, y, w, h, score) Tupel
    - person_detections: Liste von Personendetektionen als (x, y, w, h, score) Tupel
    """
    # Definieren der Fenstergrößen für Multi-Scale-Erkennung
    window_sizes = [64, 96, 128, 160, 192]
    step_size = 32
    
    # Region Proposals generieren
    regions = generate_region_proposals(image, window_sizes, step_size)
    print(f"Generierte {len(regions)} Region Proposals")
    
    # Filtern der Regionen basierend auf HOG-Features
    filtered_regions = filter_regions_by_hog(image, regions)
    print(f"Nach HOG-Filterung verbleiben {len(filtered_regions)} Regionen")
    
    # Klassifizieren der Regionen
    car_boxes = []
    car_scores = []
    person_boxes = []
    person_scores = []
    
    for i, (x, y, w, h) in enumerate(filtered_regions):
        # Region extrahieren und vorverarbeiten
        region = image.crop((x, y, x+w, y+h))
        processed_region = preprocess_image(region)
        
        # Klassifizieren der Region mit beiden Modellen
        car_prediction = car_model.predict(processed_region, verbose=0)[0][0]
        person_prediction = person_model.predict(processed_region, verbose=0)[0][0]
        
        # Regionen mit hoher Konfidenz speichern
        if car_prediction > confidence_threshold:
            car_boxes.append((x, y, w, h))
            car_scores.append(float(car_prediction))
        
        if person_prediction > confidence_threshold:
            person_boxes.append((x, y, w, h))
            person_scores.append(float(person_prediction))
    
    print(f"Nach Klassifikation verbleiben {len(car_boxes)} Auto-Regionen und {len(person_boxes)} Personen-Regionen mit Konfidenz > {confidence_threshold}")
    
    # Non-Maximum Suppression für Autos
    car_detections = []
    if len(car_boxes) > 0:
        selected_car_boxes, selected_car_scores = non_max_suppression(car_boxes, car_scores, iou_threshold)
        print(f"Nach Non-Maximum Suppression verbleiben {len(selected_car_boxes)} Auto-Detektionen")
        car_detections = [(box[0], box[1], box[2], box[3], score) for box, score in zip(selected_car_boxes, selected_car_scores)]
    else:
        print("Keine Autos erkannt")
    
    # Non-Maximum Suppression für Personen
    person_detections = []
    if len(person_boxes) > 0:
        selected_person_boxes, selected_person_scores = non_max_suppression(person_boxes, person_scores, iou_threshold)
        print(f"Nach Non-Maximum Suppression verbleiben {len(selected_person_boxes)} Personen-Detektionen")
        person_detections = [(box[0], box[1], box[2], box[3], score) for box, score in zip(selected_person_boxes, selected_person_scores)]
    else:
        print("Keine Personen erkannt")
    
    return car_detections, person_detections

## Visualisierung der Erkennungsergebnisse

Nach der Implementierung des Multi-Klassen-Objekterkennungsalgorithmus benötigen wir eine Funktion zur Visualisierung der Ergebnisse. Diese Funktion zeichnet Bounding Boxes um die erkannten Objekte und zeigt die Konfidenzwerte an, wobei verschiedene Farben für verschiedene Objektklassen verwendet werden.

Die Visualisierung ist ein wichtiger Schritt, um die Leistung des Algorithmus zu bewerten und mögliche Probleme zu identifizieren. Sie hilft uns auch, die Ergebnisse besser zu verstehen und zu kommunizieren.

In [ ]:
def visualize_multi_class_detections(image, car_detections, person_detections, output_path=None):
    """
    Visualisiert die Erkennungsergebnisse für mehrere Objektklassen.
    
    Parameter:
    - image: PIL Image-Objekt
    - car_detections: Liste von Autodetektionen als (x, y, w, h, score) Tupel
    - person_detections: Liste von Personendetektionen als (x, y, w, h, score) Tupel
    - output_path: Pfad zum Speichern des Ergebnisbildes (optional)
    
    Rückgabe:
    - result_image: PIL Image-Objekt mit gezeichneten Bounding Boxes
    """
    # Kopie des Bildes erstellen
    result_image = image.copy()
    draw = ImageDraw.Draw(result_image)
    
    # Versuchen, eine Schriftart zu laden
    try:
        font = ImageFont.truetype("arial.ttf", 16)
    except IOError:
        font = ImageFont.load_default()
    
    # Bounding Boxes für Autos zeichnen (rot)
    for x, y, w, h, score in car_detections:
        # Rechteck zeichnen
        draw.rectangle([x, y, x+w, y+h], outline="red", width=3)
        
        # Text mit Konfidenz zeichnen
        text = f"Auto: {score:.2f}"
        text_width, text_height = draw.textsize(text, font=font) if hasattr(draw, 'textsize') else (100, 20)
        draw.rectangle([x, y-text_height-4, x+text_width+4, y], fill="red")
        draw.text((x+2, y-text_height-2), text, fill="white", font=font)
    
    # Bounding Boxes für Personen zeichnen (blau)
    for x, y, w, h, score in person_detections:
        # Rechteck zeichnen
        draw.rectangle([x, y, x+w, y+h], outline="blue", width=3)
        
        # Text mit Konfidenz zeichnen
        text = f"Person: {score:.2f}"
        text_width, text_height = draw.textsize(text, font=font) if hasattr(draw, 'textsize') else (100, 20)
        draw.rectangle([x, y-text_height-4, x+text_width+4, y], fill="blue")
        draw.text((x+2, y-text_height-2), text, fill="white", font=font)
    
    # Bild speichern, falls ein Ausgabepfad angegeben wurde
    if output_path:
        result_image.save(output_path)
    
    return result_image

## Testbilder herunterladen

Bevor wir unseren Multi-Klassen-Objekterkennungsalgorithmus testen können, benötigen wir einige Testbilder. Wir laden Bilder herunter, die sowohl Personen als auch Autos enthalten, und speichern sie im Testbilder-Verzeichnis.

Die Bilder sollten verschiedene Szenarien abdecken, um die Robustheit des Algorithmus zu testen:
- Bilder mit mehreren Personen und Autos
- Bilder mit Personen und Autos in verschiedenen Größen und Perspektiven
- Bilder mit Personen und Autos in verschiedenen Umgebungen (Stadt, Straße, Parkplatz)

Diese Vielfalt an Testbildern hilft uns, die Stärken und Schwächen unseres Algorithmus zu identifizieren und zu verstehen, wie gut er in verschiedenen Szenarien funktioniert.

In [ ]:
# URLs für Testbilder mit Personen und Autos
test_image_urls = [
    "https://cdn.pixabay.com/photo/2016/11/18/12/14/car-1834274_1280.jpg",  # Personen und Autos auf der Straße
    "https://cdn.pixabay.com/photo/2017/08/01/11/48/woman-2564660_1280.jpg",  # Person neben Auto
    "https://cdn.pixabay.com/photo/2017/08/06/15/13/people-2593341_1280.jpg",  # Mehrere Personen auf der Straße mit Autos
    "https://cdn.pixabay.com/photo/2016/11/29/09/32/auto-1868726_1280.jpg",  # Person steigt in Auto ein
    "https://cdn.pixabay.com/photo/2017/08/07/23/50/car-2609551_1280.jpg"  # Familie mit Auto
]

# Testbilder herunterladen
test_images = []
test_image_paths = []

for i, url in enumerate(test_image_urls):
    try:
        # Bild herunterladen
        image = load_image_from_url(url)
        
        if image:
            # Bild speichern
            image_path = os.path.join(test_images_dir, f"test_image_person_car_{i+1}.jpg")
            image.save(image_path)
            
            # Bild und Pfad speichern
            test_images.append(image)
            test_image_paths.append(image_path)
            
            print(f"Bild {i+1} erfolgreich heruntergeladen und gespeichert unter: {image_path}")
    except Exception as e:
        print(f"Fehler beim Herunterladen von Bild {i+1}: {e}")

print(f"Insgesamt {len(test_images)} Testbilder heruntergeladen")

# Testbilder anzeigen
plt.figure(figsize=(15, 10))
for i, image in enumerate(test_images):
    plt.subplot(2, 3, i+1)
    plt.imshow(image)
    plt.title(f"Testbild {i+1}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## Anwendung der Multi-Klassen-Objekterkennung auf Testbilder

Jetzt können wir unseren Multi-Klassen-Objekterkennungsalgorithmus auf die heruntergeladenen Testbilder anwenden. Wir werden die Ergebnisse visualisieren und im Ergebnisverzeichnis speichern.

Dieser Schritt ermöglicht es uns, die Leistung des Algorithmus zu bewerten und mögliche Verbesserungen zu identifizieren. Wir können verschiedene Parameter wie den Konfidenz-Schwellenwert oder den IoU-Schwellenwert anpassen, um die Ergebnisse zu optimieren.

In [ ]:
# Multi-Klassen-Objekterkennung auf Testbildern anwenden
for i, (image, image_path) in enumerate(zip(test_images, test_image_paths)):
    print(f"\nVerarbeite Testbild {i+1}...")
    
    # Objekte erkennen
    start_time = time.time()
    car_detections, person_detections = detect_objects(image, car_model, person_model, confidence_threshold=0.7, iou_threshold=0.5)
    elapsed_time = time.time() - start_time
    
    print(f"Erkennungszeit: {elapsed_time:.2f} Sekunden")
    print(f"Erkannte {len(car_detections)} Autos und {len(person_detections)} Personen")
    
    # Ergebnisse visualisieren
    output_path = os.path.join(human_results_dir, f"result_image_person_car_{i+1}.jpg")
    result_image = visualize_multi_class_detections(image, car_detections, person_detections, output_path)
    
    # Ergebnisse anzeigen
    plt.figure(figsize=(10, 8))
    plt.imshow(result_image)
    plt.title(f"Testbild {i+1} - {len(car_detections)} Autos und {len(person_detections)} Personen erkannt")
    plt.axis('off')
    plt.show()

## Optimierung der Parameter

Die Leistung unseres Multi-Klassen-Objekterkennungsalgorithmus hängt von verschiedenen Parametern ab, die wir optimieren können. Die wichtigsten Parameter sind:

1. **Konfidenz-Schwellenwert**: Bestimmt, ab welcher Konfidenz eine Region als Auto oder Person klassifiziert wird
2. **IoU-Schwellenwert**: Bestimmt, ab welcher Überlappung Bounding Boxes als redundant betrachtet werden
3. **Fenstergrößen**: Bestimmen die Größen der Sliding Windows für die Region Proposals
4. **Schrittweite**: Bestimmt, wie weit das Sliding Window bei jedem Schritt bewegt wird

Wir können diese Parameter anpassen, um die Genauigkeit und Effizienz des Algorithmus zu verbessern. Eine niedrigere Konfidenz führt zu mehr Detektionen, aber auch zu mehr Falsch-Positiven, während ein höherer IoU-Schwellenwert zu mehr überlappenden Boxen führt.

In [ ]:
# Optimierung der Parameter
def optimize_parameters(image, car_model, person_model):
    """
    Testet verschiedene Parameter für die Multi-Klassen-Objekterkennung.
    
    Parameter:
    - image: PIL Image-Objekt
    - car_model: Trainiertes CNN-Modell für Automerkennung
    - person_model: Trainiertes CNN-Modell für Personenerkennung
    """
    # Verschiedene Konfidenz-Schwellenwerte testen
    confidence_thresholds = [0.5, 0.7, 0.9]
    
    plt.figure(figsize=(15, 10))
    for i, conf_threshold in enumerate(confidence_thresholds):
        print(f"\nTeste Konfidenz-Schwellenwert: {conf_threshold}")
        
        # Objekte erkennen
        car_detections, person_detections = detect_objects(image, car_model, person_model, confidence_threshold=conf_threshold, iou_threshold=0.5)
        
        # Ergebnisse visualisieren
        result_image = visualize_multi_class_detections(image, car_detections, person_detections)
        
        # Ergebnisse anzeigen
        plt.subplot(2, 2, i+1)
        plt.imshow(result_image)
        plt.title(f"Konfidenz > {conf_threshold} - {len(car_detections)} Autos, {len(person_detections)} Personen")
        plt.axis('off')
    
    # Verschiedene IoU-Schwellenwerte testen
    iou_threshold = 0.3
    car_detections, person_detections = detect_objects(image, car_model, person_model, confidence_threshold=0.7, iou_threshold=iou_threshold)
    result_image = visualize_multi_class_detections(image, car_detections, person_detections)
    
    plt.subplot(2, 2, 4)
    plt.imshow(result_image)
    plt.title(f"IoU > {iou_threshold} - {len(car_detections)} Autos, {len(person_detections)} Personen")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Parameter für ein Testbild optimieren
if len(test_images) > 0:
    optimize_parameters(test_images[0], car_model, person_model)  # Erstes Bild verwenden

## Zusammenfassung und Ausblick

In diesem Bonus-Notebook haben wir eine Multi-Klassen-Objekterkennung implementiert, die sowohl Personen als auch Autos in Bildern erkennen kann. Hier sind die wichtigsten Punkte:

1. **Training eines Personenerkennungsmodells**: Wir haben ein CNN-Modell trainiert, um Personen im CIFAR-10-Datensatz zu erkennen, ähnlich wie wir es für die Automerkennung getan haben.

2. **Kombination von Modellen**: Wir haben das Personenerkennungsmodell mit dem vortrainierten Autoerkennungsmodell kombiniert, um beide Objektklassen in Bildern zu erkennen.

3. **Multi-Klassen-Objekterkennung**: Wir haben einen Algorithmus implementiert, der Region Proposals generiert, beide Modelle auf jede Region anwendet und Non-Maximum Suppression für jede Klasse durchführt.

4. **Visualisierung der Ergebnisse**: Wir haben die Erkennungsergebnisse visualisiert, wobei verschiedene Farben für verschiedene Objektklassen verwendet wurden.

5. **Parameteroptimierung**: Wir haben verschiedene Parameter wie den Konfidenz-Schwellenwert und den IoU-Schwellenwert getestet, um die Leistung des Algorithmus zu optimieren.

Diese Implementierung zeigt, wie man spezialisierte Modelle für verschiedene Objektklassen kombinieren kann, um komplexere Aufgaben zu lösen. Der modulare Ansatz ermöglicht es, neue Objektklassen hinzuzufügen, ohne bestehende Modelle neu trainieren zu müssen.

Für zukünftige Verbesserungen könnten wir:
- Weitere Objektklassen hinzufügen (z.B. Fahrräder, Verkehrsschilder)
- Fortgeschrittenere Modelle wie YOLO oder Faster R-CNN verwenden
- Die Effizienz des Algorithmus verbessern, um Echtzeiterkennung zu ermöglichen
- Die Robustheit gegenüber verschiedenen Lichtbedingungen, Perspektiven und Verdeckungen verbessern

Insgesamt bietet diese Implementierung eine gute Grundlage für die Entwicklung komplexerer Objekterkennungssysteme und zeigt die Flexibilität und Erweiterbarkeit von CNN-basierten Ansätzen.